# Which temporal model is redundant — the BiLSTM or the HMM?

The architecture ablation produced a result that needs explaining before it can be acted
on: **removing the BiLSTM raises staging accuracy** from 0.7238 to 0.7386 ($p < 0.0001$)
while costing respiratory AUC 0.028.

The likely reason is that the model contains *two* mechanisms for the same job. The BiLSTM
learns stage transitions from data; the HMM imposes them through Viterbi decoding. For
staging they overlap, and the overlap appears to hurt. The respiratory head has no HMM, so
it keeps the benefit of recurrence.

Which one to remove is not a cosmetic question. Dropping the BiLSTM from the staging path
leaves staging as a per-epoch network plus a Markov smoother — accurate, but a reviewer can
fairly call it shallow. Dropping the HMM instead removes the only classical component in
the pipeline and leaves the staging path fully neural. If both reach the same accuracy, the
second is the better paper.

Four cells settle it. Because both decodings can be read from one trained model, this needs
only two configurations rather than four:

|              | HMM decoding | raw argmax |
|--------------|--------------|------------|
| **BiLSTM**   | published    | ?          |
| **no BiLSTM**| 0.7386       | ?          |

In [ ]:
import json
import os
import sys
import time

import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
CACHE = os.path.join(OUT, "temporal_vs_hmm.json")
SEEDS = [42, 1, 7]
CONFIGS = {"BiLSTM": dict(temporal="lstm"), "no BiLSTM": dict(temporal="none")}
print("device:", C.DEV, "| runs:", len(CONFIGS) * len(SEEDS))

In [ ]:
res = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
t0 = time.time()
for name, kw in CONFIGS.items():
    for seed in SEEDS:
        key = "%s|%d" % (name, seed)
        if key in res:
            continue
        r = C.run_10fold(fusion="concat", seed=seed, **kw)
        res[key] = {k: [f[k] for f in r["per_fold"]]
                    for k in ("acc", "acc_raw", "mf1", "mf1_raw",
                              "kappa", "kappa_raw", "auc", "ap")}
        json.dump(res, open(CACHE, "w"))
        print("%-10s seed %-4d  HMM %.4f   raw %.4f   AUC %.4f   (%.1f min)"
              % (name, seed, np.mean(res[key]["acc"]), np.mean(res[key]["acc_raw"]),
                 np.nanmean(res[key]["auc"]), (time.time() - t0) / 60), flush=True)
print("\ntotal %.1f min" % ((time.time() - t0) / 60))

## The 2×2

In [ ]:
def pool(name, metric):
    return np.array([v for s in SEEDS if "%s|%d" % (name, s) in res
                     for v in res["%s|%d" % (name, s)][metric]])

print("staging accuracy, mean +- fold SD over 3 seeds x 10 folds\n")
print("%-12s %-22s %-22s" % ("", "HMM decoding", "raw argmax (no HMM)"))
print("-" * 58)
cells = {}
for name in CONFIGS:
    h, r = pool(name, "acc"), pool(name, "acc_raw")
    cells[(name, "hmm")], cells[(name, "raw")] = h, r
    print("%-12s %.4f +- %.3f        %.4f +- %.3f"
          % (name, h.mean(), h.std(), r.mean(), r.std()))

print("\nrespiratory AUC (unaffected by HMM, which only decodes staging)")
for name in CONFIGS:
    u = pool(name, "auc")
    print("  %-12s %.4f +- %.3f" % (name, np.nanmean(u), np.nanstd(u)))

In [ ]:
from scipy.stats import wilcoxon

best = max(cells, key=lambda k: cells[k].mean())
print("best cell: %s + %s  ->  %.4f\n" % (best[0], best[1].upper(), cells[best].mean()))

print("%-34s %9s %9s  %s" % ("comparison", "delta", "p", "reading"))
print("-" * 76)
PAIRS = [(("BiLSTM", "hmm"), ("no BiLSTM", "hmm"), "does the BiLSTM help when HMM is present?"),
         (("BiLSTM", "hmm"), ("BiLSTM", "raw"), "does the HMM help when BiLSTM is present?"),
         (("BiLSTM", "raw"), ("no BiLSTM", "hmm"), "fully-neural vs per-epoch+HMM"),
         (("no BiLSTM", "raw"), ("no BiLSTM", "hmm"), "does the HMM help alone?")]
for a, b, why in PAIRS:
    x, y = cells[a], cells[b]
    d = y.mean() - x.mean()
    p = wilcoxon(y, x).pvalue
    print("%-34s %+9.4f %9.4f  %s"
          % ("%s/%s -> %s/%s" % (a[0], a[1], b[0], b[1]), d, p, why))

json.dump({"%s|%s" % k: dict(mean=float(v.mean()), sd=float(v.std()))
           for k, v in cells.items()},
          open(os.path.join(OUT, "temporal_vs_hmm_summary.json"), "w"), indent=1)
print("\nwrote temporal_vs_hmm_summary.json")

## Reading the result

The cell to watch is **BiLSTM + raw argmax**. If it matches or beats *no BiLSTM + HMM*, the
redundancy is the HMM's, and the right move is to delete the Viterbi smoother and keep the
recurrent decoder — a fully neural staging path that is also more accurate than the
published configuration.

If instead *no BiLSTM + HMM* remains the best cell, the recurrence really is the redundant
part. The honest response is then an asymmetric design — per-epoch representation into the
staging head with HMM decoding, recurrent context into the respiratory head — and to say
plainly in the paper that the BiLSTM does not help staging on this cohort, rather than
keeping it for appearances.